# Step 6 — Predict pIC50 for the scaffold-similarity compounds

Applies the regression models evaluated in step 2 to the 1648 compounds from the
step 4/5 scaffold similarity search (`data/Similarity_Scaffold_Compounds.csv`).

**Models.** Step 2 never saved a fitted model — it fit one per CV fold, read off the
metrics and exited. Here the two best regressors from
`results/reg_results_crossvalidation.csv` are refit on all 435 training compounds with
identical hyperparameters and `SEED = 22`, then cached to `models/`:

| model | fingerprint | CV R² | CV RMSE |
|---|---|---|---|
| SVR (primary) | morgan2 | 0.51 ± 0.06 | 0.76 |
| RF (cross-check) | morgan3 | 0.50 ± 0.07 | 0.76 |

The two are statistically indistinguishable, so both are run and their agreement is
reported per compound.

**Caveats.** CV R² ≈ 0.5 and RMSE ≈ 0.76 means a typical prediction is off by roughly
0.8 log units — usable for ranking, not for quoting a potency. That RMSE was measured
on *random* splits of the training series, so it is optimistic for new scaffolds. The
`max_sim_train` column below is the applicability-domain check; treat low-similarity
predictions as unsupported. These are triage hypotheses, not measured DRD2 activity.

In [ ]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import rdFingerprintGenerator
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

RDLogger.DisableLog("rdApp.*")

ROOT = Path.cwd().parent                      # ACMDD_D2_Receptor/
sys.path.insert(0, str(ROOT / "src"))

# Reuse the EXACT featurisation from step 2 — do not reimplement it.
from step2_convert_to_fingerprint import smiles_to_fp

SEED = 22
QUERY_CSV = Path("data/Similarity_Scaffold_Compounds.csv")
OUT_CSV = Path("data/ML_predicted_pIC50.csv")
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

# Flag predictions below this max-Tanimoto-to-training as outside the domain.
AD_CUTOFF = 0.40

## 1. Fit the regressors on all training data and cache them

Deterministic: same rows, same hyperparameters, fixed seed. Re-running reproduces the
same models bit-for-bit. Cached so later runs just load.

In [ ]:
def load_training(fp):
    """X (bits, float) and y (pIC50) from the step 2 fingerprint CSV."""
    df = pd.read_csv(ROOT / "data" / f"DopamineD2_{fp}.csv")
    bits = [c for c in df.columns if c.startswith("bit_")]
    return df[bits].to_numpy(dtype=float), df["pIC50"].to_numpy(), df["smiles"]


# Hyperparameters copied verbatim from src/step2_machine_learning_pic50.py so the
# cross-validated metrics above describe these models.
SPECS = {
    "svr_morgan2": ("morgan2", SVR(kernel="rbf", C=1.0, gamma="scale")),
    "rf_morgan3": ("morgan3", RandomForestRegressor(n_estimators=100, random_state=SEED)),
}

models, train_smiles, y_train = {}, None, None
for name, (fp, estimator) in SPECS.items():
    path = MODEL_DIR / f"reg_{name}.joblib"
    X, y_train, train_smiles = load_training(fp)
    if path.exists():
        models[name] = (fp, joblib.load(path))
        print(f"{name:12s} loaded from {path.relative_to(ROOT)}")
    else:
        estimator.fit(X, y_train)
        joblib.dump(estimator, path)
        models[name] = (fp, estimator)
        print(f"{name:12s} fit on {X.shape[0]} compounds ({fp}) -> {path.relative_to(ROOT)}")

print(f"\nTraining set: {len(train_smiles)} compounds, "
      f"pIC50 {y_train.min():.2f}-{y_train.max():.2f}")

## 2. Load the query compounds

The `alignment` column holds `"<SMILES> <id>"`, so it is split on whitespace.

In [ ]:
query = pd.read_csv(QUERY_CSV)
query["smiles"] = query["alignment"].str.split().str[0]
query["compound_id"] = query["alignment"].str.split().str[1]

valid = query["smiles"].apply(lambda s: Chem.MolFromSmiles(s) is not None)
if not valid.all():
    print(f"Dropping {(~valid).sum()} unparseable SMILES")
query = query[valid].reset_index(drop=True)

n_overlap = query["smiles"].isin(set(train_smiles)).sum()
print(f"{len(query)} compounds to predict ({query['smiles'].nunique()} unique)")
print(f"Already in the training set: {n_overlap}")
query[["compound_id", "smiles", "ecfp4", "daylight"]].head()

## 3. Featurise and predict

Each model is fed the fingerprint it was trained on, built by the same
`smiles_to_fp` the training data used. A mismatch here produces confident nonsense.

In [ ]:
for name, (fp, model) in models.items():
    X_new = np.stack([smiles_to_fp(s, fp) for s in query["smiles"]]).astype(float)
    query[f"pred_pIC50_{name}"] = model.predict(X_new)
    print(f"{name:12s} {fp:8s} X = {X_new.shape}")

pred_cols = [c for c in query.columns if c.startswith("pred_pIC50_")]
query["pred_pIC50_mean"] = query[pred_cols].mean(axis=1)
query["model_spread"] = query[pred_cols].max(axis=1) - query[pred_cols].min(axis=1)
query[pred_cols + ["pred_pIC50_mean", "model_spread"]].describe().round(2)

## 4. Applicability domain

Max Tanimoto similarity (Morgan-2) of each query compound to any training compound.
A model interpolates; it cannot extrapolate to chemistry it never saw.

In [ ]:
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
train_fps = [gen.GetFingerprint(Chem.MolFromSmiles(s)) for s in train_smiles]

query["max_sim_train"] = [
    max(DataStructs.BulkTanimotoSimilarity(gen.GetFingerprint(Chem.MolFromSmiles(s)), train_fps))
    for s in query["smiles"]
]
query["in_domain"] = query["max_sim_train"] >= AD_CUTOFF

print(query["max_sim_train"].describe().round(3).to_string())
print(f"\nIn domain (max_sim_train >= {AD_CUTOFF}): "
      f"{query['in_domain'].sum()} / {len(query)}")

## 5. Results

In [ ]:
cols = ["compound_id", "smiles", "pred_pIC50_mean"] + pred_cols + [
    "model_spread", "max_sim_train", "in_domain", "ecfp4", "daylight", "dist",
]
results = query[cols].sort_values("pred_pIC50_mean", ascending=False).reset_index(drop=True)
results.to_csv(OUT_CSV, index=False)
print(f"Wrote {len(results)} predictions to {OUT_CSV}")

print("\nTop 15 in-domain compounds by predicted pIC50:")
results[results["in_domain"]].head(15).round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(y_train, bins=25, alpha=0.6, density=True, label=f"training (n={len(y_train)})")
axes[0].hist(results["pred_pIC50_mean"], bins=25, alpha=0.6, density=True,
             label=f"predicted (n={len(results)})")
axes[0].set_xlabel("pIC50")
axes[0].set_ylabel("density")
axes[0].set_title("Predicted vs. training distribution")
axes[0].legend()

axes[1].scatter(results[pred_cols[0]], results[pred_cols[1]], s=8, alpha=0.4)
lims = [results[pred_cols].min().min(), results[pred_cols].max().max()]
axes[1].plot(lims, lims, "r--", lw=1)
axes[1].set_xlabel(pred_cols[0].replace("pred_pIC50_", ""))
axes[1].set_ylabel(pred_cols[1].replace("pred_pIC50_", ""))
axes[1].set_title("Model agreement")

sc = axes[2].scatter(results["max_sim_train"], results["pred_pIC50_mean"],
                     c=results["model_spread"], s=8, cmap="viridis")
axes[2].axvline(AD_CUTOFF, color="r", ls="--", lw=1)
axes[2].set_xlabel("max Tanimoto to training set")
axes[2].set_ylabel("predicted pIC50 (mean)")
axes[2].set_title("Prediction vs. applicability domain")
fig.colorbar(sc, ax=axes[2], label="model spread")

fig.tight_layout()
fig.savefig("data/predicted_pIC50_scaffold_compounds.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Reading these numbers

- **Rank, don't quote.** With CV RMSE ≈ 0.76, a predicted 8.4 vs. 7.9 is not a
  meaningful difference. Use the ordering to choose what to dock or assay next.
- **Expect regression to the mean.** Both models are trained on 435 compounds spanning
  pIC50 4.0–9.5 and will compress predictions toward the training mean, so the
  predicted range is narrower than the training range. A high prediction means
  "resembles the actives", not "will be this potent".
- **Check `max_sim_train` before `pred_pIC50_mean`.** These compounds came from a
  similarity search seeded on one cluster, so they are a narrow slice of chemistry and
  many may sit near the domain edge.
- **`model_spread` is a cheap uncertainty proxy.** Where SVR and RF disagree by more
  than ~0.5 log units, neither is anchored by nearby training data.
- **Keep provenance separate.** These are ML predictions. Do not merge them into the
  experimental IC50/Kd tables or compare them directly with docking scores.